In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("employee.csv")
df = df.drop(columns=['Unnamed: 8', 'Unnamed: 9'])
df.head()

,employee_id,name,department,hire_date,salary,performance_rating,email,location
0,1305.0,Jane Young,Operations,05/04/2018,NaN,3.0,jane.young@company.com,Chicago
1,1500.0,Barbara Thompson,hr,22/03/2019,"$53,879.00",2.0,barbara.thompson@company.com,chicago
2,1442.0,Lisa Jones,Sales,28/10/2015,NaN,1.0,lisa.jones@company.com,Chicago
3,1154.0,lisa Williams,ENGINEERING,09/05/2017,NaN,2.0,lisa.williams@company.com,boston
4,1448.0,Daniel Jackson,Human Resources,02/02/2016,73244,4.0,daniel.jackson@company.com,boston


In [3]:
df['name'] = df['name'].str.strip()
df['name'] = df['name'].str.replace('  ', ' ', regex=False)
df['name'] = df['name'].str.title()

print(df['name'].head(20))
print('Double spaces remaining:', df['name'].str.contains('  ', na=False).sum())

0            Jane Young
1      Barbara Thompson
2            Lisa Jones
3         Lisa Williams
4        Daniel Jackson
5         Karen Johnson
6        Linda Williams
7       Andrew Martinez
8        Patricia Jones
9           Lisa Martin
10      James Hernandez
11          Sarah Clark
12         Sarah Wilson
13           Mary Young
14       Susan Martinez
15      Daniel Anderson
16    Christopher Lopez
17     Barbara Thompson
18      Elizabeth Young
19          Kevin Moore
Name: name, dtype: object
Double spaces remaining: 0


In [4]:
df['email'] = df['email'].str.lower()
def generate_email(row):
    if pd.isna(row['email']):
        name = row['name'].lower().replace(' ', '.')
        return f"{name}@company.com"
    return row['email']

df['email'] = df.apply(generate_email, axis=1)
print('Null emails remaining:', df['email'].isnull().sum())
print(df['email'].head(20))

Null emails remaining: 0
0            jane.young@company.com
1      barbara.thompson@company.com
2            lisa.jones@company.com
3         lisa.williams@company.com
4        daniel.jackson@company.com
5         karen.johnson@company.com
6        linda.williams@company.com
7       andrew.martinez@company.com
8        patricia.jones@company.com
9           lisa.martin@company.com
10      james.hernandez@company.com
11          sarah.clark@company.com
12         sarah.wilson@company.com
13           mary.young@company.com
14       susan.martinez@company.com
15      daniel.anderson@company.com
16    christopher.lopez@company.com
17     barbara.thompson@company.com
18      elizabeth.young@company.com
19          kevin.moore@company.com
Name: email, dtype: object


In [5]:
def clean_dept(dept):
    if pd.isna(dept):
        return np.nan
    dept = dept.strip().lower()
    if dept in ['hr', 'h.r.', 'human resources']:
        return 'HR'
    elif dept in ['sales', 'sales ']:
        return 'Sales'
    elif dept in ['engineering', 'eng']:
        return 'Engineering'
    elif dept in ['it', 'i.t.', 'information technology']:
        return 'IT'
    elif dept in ['finance', 'fin']:
        return 'Finance'
    elif dept in ['operations', 'ops']:
        return 'Operations'
    elif dept in ['marketing']:
        return 'Marketing'
    else:
        return np.nan

df['department'] = df['department'].apply(clean_dept)
print(df['department'].unique())

['Operations' 'HR' 'Sales' 'Engineering' 'Finance' 'IT' 'Marketing']


In [6]:
def clean_location(loc):
    if pd.isna(loc):
        return np.nan
    loc = loc.strip().lower()
    if loc in ['chicago']:
        return 'Chicago'
    elif loc in ['new york', 'ny', 'new york ']:
        return 'New York'
    elif loc in ['san francisco', 'sf', 'san fran']:
        return 'San Francisco'
    elif loc in ['boston']:
        return 'Boston'
    elif loc in ['austin']:
        return 'Austin'
    else:
        return np.nan

df['location'] = df['location'].apply(clean_location)
print(df['location'].unique())

['Chicago' 'Boston' 'San Francisco' 'New York' 'Austin']


In [7]:
# Remove $ and commas
df['salary'] = df['salary'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')

# Replace outliers (above 120,000 or below 0) with NaN
df.loc[df['salary'] > 120000, 'salary'] = np.nan
df.loc[df['salary'] < 0, 'salary'] = np.nan

# Fill NaN salary with department average
dept_salary_avg = df.groupby('department')['salary'].transform('mean')
df['salary'] = df['salary'].fillna(dept_salary_avg).round(2)

print('Null salaries remaining:', df['salary'].isnull().sum())
print('Min salary:', df['salary'].min())
print('Max salary:', df['salary'].max())

Null salaries remaining: 0
Min salary: 41025.0
Max salary: 119873.0


In [8]:
df['hire_date'] = pd.to_datetime(df['hire_date'], dayfirst=True, errors='coerce')
df['hire_date'] = df['hire_date'].dt.strftime('%Y-%m-%d')
print(df['hire_date'].head(10))
print('Null hire dates:', df['hire_date'].isnull().sum())

0    2018-04-05
1    2019-03-22
2    2015-10-28
3    2017-05-09
4    2016-02-02
5           NaN
6    2020-07-18
7    2016-06-10
8           NaN
9           NaN
Name: hire_date, dtype: object
Null hire dates: 99


In [9]:
# Fill missing performance ratings with median
df['performance_rating'] = df['performance_rating'].fillna(df['performance_rating'].median())

# Fill missing employee IDs
df['employee_id'] = df['employee_id'].ffill()

# Fill any remaining nulls in department and location
df['department'] = df['department'].fillna('Engineering')
df['location'] = df['location'].fillna('Unknown')

print(df.isnull().sum())

employee_id            0
name                   0
department             0
hire_date             99
salary                 0
performance_rating     0
email                  0
location               0
dtype: int64


In [10]:
print('Shape:', df.shape)
print()
print('Null counts:')
print(df.isnull().sum())
print()
print('Department unique:', df['department'].unique())
print('Location unique:', df['location'].unique())
print()
df.head(20)

Shape: (515, 8)

Null counts:
employee_id            0
name                   0
department             0
hire_date             99
salary                 0
performance_rating     0
email                  0
location               0
dtype: int64

Department unique: ['Operations' 'HR' 'Sales' 'Engineering' 'Finance' 'IT' 'Marketing']
Location unique: ['Chicago' 'Boston' 'San Francisco' 'New York' 'Austin']



,employee_id,name,department,hire_date,salary,performance_rating,email,location
0,1305.0,Jane Young,Operations,2018-04-05,57992.60,3.0,jane.young@company.com,Chicago
1,1500.0,Barbara Thompson,HR,2019-03-22,53879.00,2.0,barbara.thompson@company.com,Chicago
2,1442.0,Lisa Jones,Sales,2015-10-28,70697.70,1.0,lisa.jones@company.com,Chicago
3,1154.0,Lisa Williams,Engineering,2017-05-09,95123.98,2.0,lisa.williams@company.com,Boston
4,1448.0,Daniel Jackson,HR,2016-02-02,73244.00,4.0,daniel.jackson@company.com,Boston
5,1132.0,Karen Johnson,Operations,NaN,43190.00,3.0,karen.johnson@company.com,San Francisco
6,1205.0,Linda Williams,HR,2020-07-18,62593.00,5.0,linda.williams@company.com,San Francisco
7,1276.0,Andrew Martinez,HR,2016-06-10,77022.00,2.0,andrew.martinez@company.com,Chicago
8,1326.0,Patricia Jones,HR,NaN,60792.00,3.0,patricia.jones@company.com,New York
9,1248.0,Lisa Martin,HR,NaN,58312.00,5.0,lisa.martin@company.com,New York


In [11]:
dept_table = df[['department']].drop_duplicates().reset_index(drop=True)
dept_table.insert(0, 'dept_id', range(1, len(dept_table) + 1))
print(dept_table)

   dept_id   department
0        1   Operations
1        2           HR
2        3        Sales
3        4  Engineering
4        5      Finance
5        6           IT
6        7    Marketing


In [12]:
location_table = df[['location']].drop_duplicates().reset_index(drop=True)
location_table.insert(0, 'location_id', range(1, len(location_table) + 1))
print(location_table)

   location_id       location
0            1        Chicago
1            2         Boston
2            3  San Francisco
3            4       New York
4            5         Austin


In [13]:
employee_table = df.merge(dept_table, on='department').merge(location_table, on='location')
employee_table = employee_table[['employee_id', 'name', 'dept_id', 'hire_date', 'salary', 'performance_rating', 'email', 'location_id']]
print(employee_table.shape)
print(employee_table.head(10))

(515, 8)
   employee_id              name  dept_id   hire_date    salary  \
0       1305.0        Jane Young        1  2018-04-05  57992.60   
1       1500.0  Barbara Thompson        2  2019-03-22  53879.00   
2       1442.0        Lisa Jones        3  2015-10-28  70697.70   
3       1154.0     Lisa Williams        4  2017-05-09  95123.98   
4       1448.0    Daniel Jackson        2  2016-02-02  73244.00   
5       1132.0     Karen Johnson        1         NaN  43190.00   
6       1205.0    Linda Williams        2  2020-07-18  62593.00   
7       1276.0   Andrew Martinez        2  2016-06-10  77022.00   
8       1326.0    Patricia Jones        2         NaN  60792.00   
9       1248.0       Lisa Martin        2         NaN  58312.00   

   performance_rating                         email  location_id  
0                 3.0        jane.young@company.com            1  
1                 2.0  barbara.thompson@company.com            1  
2                 1.0        lisa.jones@company.com 

In [14]:
dept_table.to_csv("department_table.csv", index=False)

In [15]:
location_table.to_csv("location_table.csv", index=False)

In [16]:
employee_table.to_csv("employee_table.csv", index=False)

In [17]:
print("All 3 tables saved successfully!")

All 3 tables saved successfully!


In [18]:
import os
print(os.getcwd())
print(os.listdir())

/Users/olaoluwaadekanye
['Cafe_Sales_Report.xlsx', 'edb_sqlprotect_pg16.app.zip', '.config', 'Music', '.zprofile.pysave', '.copilot', '.windsurf', '.codeium', 'Untitled5.ipynb', 'CAFE_SALES_FINAL.csv', 'Untitled1.ipynb', '.DS_Store', 'Cafe_Sales_Final_Cleaned.xlsx', '.CFUserTextEncoding', 'monthly_trend.png', '.xonshrc', 'employee_data_CLEANED_FINAL.csv', 'anaconda_projects', 'Untitled3.ipynb', 'consumer_clean.csv', 'Untitled.ipynb', 'employee.csv', 'plugins', '.zshrc', 'Untitled4.ipynb', 'Pictures', 'location_table.csv', '.zprofile', 'consumer_clean.xlsx', 'consumer_cleaning.csv', 'timely_response.png', '.claude', 'employee_table.csv', '.zsh_history', 'Untitled2.ipynb', '.ipython', 'Desktop', 'Library', '.matplotlib', 'CAFE_SALES.ipynb', 'CAFE_SALES_CLEANED(in).csv', 'Employeedataset.ipynb', 'complaints_by_product.png', 'Public', '.idlerc', '.tcshrc', 'metabase.db.trace.db', '.virtual_documents', '.anaconda', 'Movies', 'Applications', 'employee_data_messy1_clean_employee_data_.csv', '

In [19]:
df = pd.read_csv("/Users/olaoluwaadekanye/Downloads/employee.csv")

In [20]:
import pandas as pd
df = pd.read_csv("/Users/olaoluwaadekanye/Downloads/employee.csv")
print(df.shape)
print(df.head())

(515, 10)
   employee_id              name       department   hire_date      salary  \
0       1305.0        Jane Young       Operations  05/04/2018         NaN   
1       1500.0  Barbara Thompson               hr  22/03/2019  $53,879.00   
2       1442.0       Lisa  Jones           Sales   28/10/2015         NaN   
3       1154.0     lisa Williams      ENGINEERING  09/05/2017         NaN   
4       1448.0    Daniel Jackson  Human Resources  02/02/2016       73244   

   performance_rating                         email  location  Unnamed: 8  \
0                 3.0        jane.young@company.com   Chicago         NaN   
1                 2.0  barbara.thompson@company.com   chicago         NaN   
2                 1.0        lisa.jones@company.com   Chicago         NaN   
3                 2.0     lisa.williams@company.com    boston         NaN   
4                 4.0    daniel.jackson@company.com    boston         NaN   

   Unnamed: 9  
0         NaN  
1         NaN  
2         NaN  


In [21]:
import pandas as pd
import numpy as np
import os

# Load file
df = pd.read_csv("/Users/olaoluwaadekanye/Downloads/employee.csv")
df = df.drop(columns=['Unnamed: 8', 'Unnamed: 9'])

# Clean names
df['name'] = df['name'].str.strip().str.replace('  ', ' ', regex=False).str.title()

# Clean department
def clean_dept(dept):
    if pd.isna(dept):
        return np.nan
    dept = dept.strip().lower()
    if dept in ['hr', 'h.r.', 'human resources']:
        return 'HR'
    elif dept in ['sales', 'sales ']:
        return 'Sales'
    elif dept in ['engineering', 'eng']:
        return 'Engineering'
    elif dept in ['it', 'i.t.', 'information technology']:
        return 'IT'
    elif dept in ['finance', 'fin']:
        return 'Finance'
    elif dept in ['operations', 'ops']:
        return 'Operations'
    elif dept in ['marketing']:
        return 'Marketing'
    else:
        return np.nan

df['department'] = df['department'].apply(clean_dept)

# Clean location
def clean_location(loc):
    if pd.isna(loc):
        return np.nan
    loc = loc.strip().lower()
    if loc in ['chicago']:
        return 'Chicago'
    elif loc in ['new york', 'ny', 'new york ']:
        return 'New York'
    elif loc in ['san francisco', 'sf', 'san fran']:
        return 'San Francisco'
    elif loc in ['boston']:
        return 'Boston'
    elif loc in ['austin']:
        return 'Austin'
    else:
        return np.nan

df['location'] = df['location'].apply(clean_location)

# Clean salary
df['salary'] = df['salary'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df.loc[df['salary'] > 120000, 'salary'] = np.nan
df.loc[df['salary'] < 0, 'salary'] = np.nan
dept_salary_avg = df.groupby('department')['salary'].transform('mean')
df['salary'] = df['salary'].fillna(dept_salary_avg).round(2)

# Clean email
df['email'] = df['email'].str.lower()
def generate_email(row):
    if pd.isna(row['email']):
        name = row['name'].lower().replace(' ', '.')
        return f"{name}@company.com"
    return row['email']
df['email'] = df.apply(generate_email, axis=1)

# Fix hire date
df['hire_date'] = pd.to_datetime(df['hire_date'], dayfirst=True, errors='coerce')
df['hire_date'] = df['hire_date'].dt.strftime('%Y-%m-%d')

# Fix remaining nulls
df['performance_rating'] = df['performance_rating'].fillna(df['performance_rating'].median())
df['employee_id'] = df['employee_id'].ffill()
df['department'] = df['department'].fillna('Engineering')
df['location'] = df['location'].fillna('Unknown')

In [22]:
# Create 3 tables
dept_table = df[['department']].drop_duplicates().reset_index(drop=True)
dept_table.insert(0, 'dept_id', range(1, len(dept_table) + 1))

location_table = df[['location']].drop_duplicates().reset_index(drop=True)
location_table.insert(0, 'location_id', range(1, len(location_table) + 1))

employee_table = df.merge(dept_table, on='department').merge(location_table, on='location')
employee_table = employee_table[['employee_id', 'name', 'dept_id', 'hire_date', 'salary', 'performance_rating', 'email', 'location_id']]

# Save to Downloads folder
downloads = "/Users/olaoluwaadekanye/Downloads"
dept_table.to_csv(f"{downloads}/department_table.csv", index=False)
location_table.to_csv(f"{downloads}/location_table.csv", index=False)
employee_table.to_csv(f"{downloads}/employee_table.csv", index=False)

print("✅ All done!")
print("Department rows:", len(dept_table))
print("Location rows:", len(location_table))
print("Employee rows:", len(employee_table))
print("Saved to:", downloads)

✅ All done!
Department rows: 7
Location rows: 5
Employee rows: 515
Saved to: /Users/olaoluwaadekanye/Downloads


In [23]:
import mysql.connector
import pandas as pd
import numpy as np

# Clean employee table
employee_clean = employee_table.copy()
employee_clean = employee_clean.where(pd.notnull(employee_clean), None)

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Adejumoke123",
    database="employee_db"
)
cursor = conn.cursor()

success = 0
failed = 0

for _, row in employee_clean.iterrows():
    try:
        cursor.execute(
            "INSERT IGNORE INTO employee VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
            (
                None if pd.isna(row['employee_id']) else int(row['employee_id']),
                row['name'],
                None if pd.isna(row['dept_id']) else int(row['dept_id']),
                row['hire_date'],
                None if pd.isna(row['salary']) else float(row['salary']),
                None if pd.isna(row['performance_rating']) else int(row['performance_rating']),
                row['email'],
                None if pd.isna(row['location_id']) else int(row['location_id'])
            )
        )
        success += 1
    except Exception as e:
        failed += 1
        print(f"Row {_} failed: {e}")
        print(f"Row data: {row.to_dict()}")
        break

conn.commit()
print(f"✅ Success: {success} rows")
print(f"❌ Failed: {failed} rows")

cursor.close()
conn.close()

Row 0 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row data: {'employee_id': 1305.0, 'name': 'Jane Young', 'dept_id': 1, 'hire_date': '2018-04-05', 'salary': 57992.6, 'performance_rating': 3.0, 'email': 'jane.young@company.com', 'location_id': 1}
✅ Success: 0 rows
❌ Failed: 1 rows


In [24]:
import mysql.connector
import pandas as pd

# Clean employee table
employee_clean = employee_table.copy()
employee_clean = employee_clean.where(pd.notnull(employee_clean), None)

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Adejumoke123",
    database="employee_db",
    connection_timeout=180
)

# Clear any existing locks
cursor = conn.cursor()
cursor.execute("SET innodb_lock_wait_timeout = 120")

success = 0
failed = 0

for _, row in employee_clean.iterrows():
    try:
        cursor.execute(
            "INSERT IGNORE INTO employee VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
            (
                None if pd.isna(row['employee_id']) else int(row['employee_id']),
                row['name'],
                None if pd.isna(row['dept_id']) else int(row['dept_id']),
                row['hire_date'],
                None if pd.isna(row['salary']) else float(row['salary']),
                None if pd.isna(row['performance_rating']) else int(row['performance_rating']),
                row['email'],
                None if pd.isna(row['location_id']) else int(row['location_id'])
            )
        )
        success += 1
    except Exception as e:
        failed += 1
        print(f"Row {_} failed: {e}")

conn.commit()
print(f"✅ Success: {success} rows")
print(f"❌ Failed: {failed} rows")

cursor.close()
conn.close()

Row 0 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 1 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 2 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 3 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 4 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 5 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 6 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 7 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 8 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 9 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 10 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 11 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 12 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 13 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Ro

In [25]:
import mysql.connector
import pandas as pd

employee_clean = employee_table.copy()
employee_clean = employee_clean.where(pd.notnull(employee_clean), None)

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Adejumoke123",
    database="employee_db",
    connection_timeout=180,
    autocommit=True
)

cursor = conn.cursor()
success = 0
failed = 0

# Insert department
for _, row in dept_table.iterrows():
    cursor.execute(
        "INSERT IGNORE INTO department VALUES (%s, %s)",
        (int(row['dept_id']), row['department'])
    )
print("✅ Department done")

# Insert location
for _, row in location_table.iterrows():
    cursor.execute(
        "INSERT IGNORE INTO location VALUES (%s, %s)",
        (int(row['location_id']), row['location'])
    )
print("✅ Location done")

# Insert employee
for _, row in employee_clean.iterrows():
    try:
        cursor.execute(
            "INSERT IGNORE INTO employee VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
            (
                None if pd.isna(row['employee_id']) else int(row['employee_id']),
                row['name'],
                None if pd.isna(row['dept_id']) else int(row['dept_id']),
                row['hire_date'],
                None if pd.isna(row['salary']) else float(row['salary']),
                None if pd.isna(row['performance_rating']) else int(row['performance_rating']),
                row['email'],
                None if pd.isna(row['location_id']) else int(row['location_id'])
            )
        )
        success += 1
    except Exception as e:
        failed += 1
        print(f"Row {_} failed: {e}")

print(f"✅ Employee: {success} rows inserted")
print(f"❌ Failed: {failed} rows")
cursor.close()
conn.close()
print("🎉 All done!")

✅ Department done
✅ Location done
Row 0 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 1 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 2 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 3 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 4 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 5 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 6 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 7 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 8 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 9 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 10 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 11 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 12 failed: 1146 (42S02): Table 'employee_db.employee' doesn't exist
Row 13 failed: 1146 (42S02): Table 'empl

In [26]:
# Check for duplicates
print("Duplicates:", employee_table.duplicated(subset=['employee_id']).sum())

# Remove duplicates
employee_table_clean = employee_table.drop_duplicates(subset=['employee_id'], keep='first')
print("Original rows:", len(employee_table))
print("After removing duplicates:", len(employee_table_clean))

# Save to Downloads
employee_table_clean.to_csv("/Users/olaoluwaadekanye/Downloads/employee_table_final.csv", index=False)
print("✅ Saved!")

Duplicates: 20
Original rows: 515
After removing duplicates: 495
✅ Saved!
